In [ ]:
from pde_backtest import PDEBacktest, long_short_analysis
from config_loader import config
from data_fetcher import data_fetcher
from strats import _build_p_base
from models import MLEModel, OLSModel, NULLModel
from strategies import STRATEGY_REGISTRY

import numpy as np
import pandas as pd


In [ ]:
lots = config.get('trading','lots')
symbols = ['BTCUSD' ]#, 'ETHUSD', 'XAUUSD', 'XAGUSD']
tf = ['M1']#, 'M5', 'M15', 'H1']
data_fetcher.connect()
dfs = {
    f"{tf_name}_{s}": data_fetcher.fetch_data(tf_name, symbol=s, bars_count=80000) 
    for tf_name in tf 
    for s in symbols
}


In [ ]:
for key,df in dfs.items():
    print(f"{key}: {len(df)} rows")

In [ ]:
tps = [5,10,20]
tpsl_ratio = [1,1.2,1.4,1.6,1.8,2,2.5,3,4]
sls = {tp:[tp/r for r in tpsl_ratio] for tp in tps}

recompute_thresh = [0.2,0.4,1,4,8,16,30]
MODEL_REGISTRY = {
    'OLS':OLSModel,
    'MLE':MLEModel,
    'NULL':NULLModel
}
STRATEGY_REGISTRY

In [ ]:
model_params = {
    'ols_window': range(1,200,5),
    'mle_window': range(1,200,5),
    'column' : ['open', 'close']
}

In [ ]:
strategy_params = {
  "pde_edge":{
    "edge_threshold_long" : np.linspace(-1,1,20),
    "edge_threshold_short" : np.linspace(-1,1,20),
  },
  "vol_regime":{
    "vol_ratio_threshold" : np.linspace(0.1,2,20),
  },
  "pde_vol_mu":{
    "mu_threshold_long" : np.linspace(0,1,10),
    "mu_threshold_short" : np.linspace(-1,0,10),
    "edge_threshold" : np.linspace(-1,1,20),
    "vol_max" : np.linspace(0.1,2,20),
    "vol_strong" : np.linspace(0.1,2,20),
    "p_threshold" :np.linspace(0.0,1,20)
  },
  "always_buy" : {}
}

In [ ]:
import numpy as np
from dataclasses import dataclass

@dataclass
class Fold:
    fold_idx:   int
    train_idx:  np.ndarray
    test_idx:   np.ndarray

def purged_kfold(n_bars: int, n_splits: int = 5,
                 purge_window: int = 200,
                 embargo_window: int = 50) -> list[Fold]:
    idx       = np.arange(n_bars)
    fold_size = (n_bars - purge_window) // n_splits
    folds     = []

    for k in range(n_splits):
        test_start = purge_window + k * fold_size
        test_end   = test_start + fold_size if k < n_splits - 1 else n_bars
        train_end  = test_start - purge_window
        if train_end <= 0:
            continue
        folds.append(Fold(
            fold_idx  = k + 1,
            train_idx = idx[:train_end],
            test_idx  = idx[test_start:test_end],
        ))

    return folds

In [ ]:
purged_cross_valid_dataset = {}
for key,df in dfs.items():
    folds = purged_kfold(n_bars=len(df), n_splits=6, purge_window=500, embargo_window=50)
    for fold in folds:
        df_train = df.iloc[fold.train_idx].copy()
        df_test  = df.iloc[fold.test_idx].copy()
        purged_cross_valid_dataset[key] = purged_cross_valid_dataset.get(key, []) + [(df_train, df_test)]


In [ ]:
for key in purged_cross_valid_dataset.keys():
    print(
        key," Folds:",len(purged_cross_valid_dataset[key]),
        " Per Fold (train, test): ",len(purged_cross_valid_dataset[key][0]))

In [ ]:
del dfs

# Optimization
----

- Grid search for best sharpe and calmer with PnL on train set
- Based on top K parameters, run on test set

In [ ]:
import pickle
from pathlib import Path

def sample_params(model_params: dict, strategy_params: dict,
                  tps, sls, thresh, n_samples: int,
                  seed: int = None,
                  seen_path: str = "seen_params.pkl"):
    rng    = np.random.default_rng(seed)
    models = list(MODEL_REGISTRY.keys())
    strats = list(STRATEGY_REGISTRY.keys())

    # load existing seen set if resuming
    seen_file = Path(seen_path)
    seen      = pickle.loads(seen_file.read_bytes()) if seen_file.exists() else set()
    print(f"  Resuming from {len(seen)} already-seen configs")

    attempts     = 0
    max_attempts = n_samples * 10
    generated    = 0

    while generated < n_samples and attempts < max_attempts:
        attempts += 1
        strategy = rng.choice(strats)
        model    = rng.choice(models)
        tp       = rng.choice(tps)
        sl       = rng.choice(sls[tp])
        th       = rng.choice(thresh)
        params   = {
            "strategy" : strategy,
            "model"    : model,
            "tp"       : tp,
            "sl"       : sl,
            "thresh"   : th,
            **{k: rng.choice(v) for k, v in model_params.items()},
            **{k: rng.choice(v) for k, v in strategy_params[strategy].items()},
        }
        for k, v in params.items():
            if isinstance(v, dict):
                print(f"  dict value at key '{k}': {v}")
        key = frozenset((k, round(v, 6) if isinstance(v, float) else v)
                        for k, v in params.items())
        if key in seen:
            continue

        seen.add(key)
        seen_file.write_bytes(pickle.dumps(seen))  # persist after every new sample
        generated += 1
        yield params

In [ ]:
sample_param_gen = sample_params(model_params, strategy_params, tps, sls, recompute_thresh, n_samples=50000, seed=42)

In [ ]:
key = 'M1_XAUUSD'
train_test = 0 # 0 train 1 test
params = sample_param_gen.__next__()

for fold in range(len(purged_cross_valid_dataset[key])):
    df = purged_cross_valid_dataset[key][fold][train_test]
    p_base = _build_p_base(params['tp'], params['sl'])
    bt = PDEBacktest(
        p_base, 
        MODEL_REGISTRY[params['model']](), 
        params['thresh'],
        lots=0.01,
        symbol='',
        strategy=STRATEGY_REGISTRY[params['strategy']](),
        config=params
    )
    bt.run(df)
    trades_df = bt.trades_df.copy()
    del bt

In [ ]:
long_short_analysis(bt)